In [0]:
%run /Users/arnaldo.jbj@gmail.com/ElectricalGeneratedDistribution-Project/step00_IMPORTS

In [0]:
%run /Users/arnaldo.jbj@gmail.com/ElectricalGeneratedDistribution-Project/step00_AUXILIARY_FUNCTIONS

We have tables for each (regional, month). The idea is combine all regional's mensuration in a unique file per month after promote corrections

In [0]:
list_regional = ['co', 'nordeste', 'sul', 'sudeste']

list_months = sorted(
                        set(
                               row.tableName.split('_')[2]
                               for row in spark.sql("SHOW TABLES IN bronze").collect()
                               if row.tableName.startswith('medicoes_')
                           )
                    )

month_map = F.create_map(
                          F.lit('jan'), F.lit('01'), F.lit('fev'), F.lit('02'),
                          F.lit('mar'), F.lit('03'), F.lit('abr'), F.lit('04'),
                          F.lit('mai'), F.lit('05'), F.lit('jun'), F.lit('06'),
                          F.lit('jul'), F.lit('07'), F.lit('ago'), F.lit('08'),
                          F.lit('set'), F.lit('09'), F.lit('out'), F.lit('10'),
                          F.lit('nov'), F.lit('11'), F.lit('dez'), F.lit('12'),
                        )

In [0]:
def _extract_version(t):
    """Extract version number from a versioned table name (e.g., medicoes_co_202510_v2 → 2)."""
    parts = t.rsplit("_v", 1)
    return int(parts[1]) if len(parts) > 1 and parts[1].isdigit() else 0


invoice_seq_offset = 0  # running offset — keeps invoice_id numbering continuous across months

for m in list_months:  
    df_month = None
    for reg in list_regional:
        base_name = f"medicoes_{reg}_{m}"

        # Find all matching tables, including versioned variants (e.g., _v2, _v3)
        all_bronze_tables = [r.tableName for r in spark.sql("SHOW TABLES IN bronze").collect()]
        matching = [t for t in all_bronze_tables if t == base_name or t.startswith(base_name + "_v")]
        if not matching:
            print(f"bronze.{base_name} not found — skipping")
            continue

        # Keeping latest versioned (if exists)
        versioned = [t for t in matching if t != base_name]
        if versioned:
            table_origin = sorted(versioned, key=_extract_version, reverse=True)[0]
            if base_name in matching:
                print(f"{base_name} superseded by {table_origin}")
        else:
            table_origin = base_name

        # Reading table
        df_bronze = spark.read.table(f"bronze.{table_origin}")

        if 'Unidade_Consumidora' in df_bronze.columns:
            # ---- CSV format (regional: nordeste) ----
            df = (
                  df_bronze.withColumnsRenamed(
                                                {
                                                  'Unidade_Consumidora': 'uc',
                                                  'Mês_Referência': 'generated_at',
                                                  'Data_Leitura': 'reading_at',
                                                  'Consumo__MWh_': 'consume_mwh',
                                                  'Energia_Compensada__MWh_': 'compensated_energy_mwh',
                                                  'Saldo_Créditos__MWh_': 'balance_mwh',
                                                }
                                              )
                           .withColumn(
                                        'generated_at',
                                        F.date_format(
                                                       F.to_date(
                                                                  F.concat(
                                                                            F.lit('20'),
                                                                            F.element_at(F.split(F.col('generated_at'), '/'), 2),
                                                                            F.lit('-'),
                                                                            month_map[F.element_at(F.split(F.col('generated_at'), '/'), 1)],
                                                                            F.lit('-01')
                                                                          ),
                                                                  'yyyy-MM-dd'
                                                                ),
                                                       'yyyy-MM-dd'
                                                     )
                                      )
                           .withColumn(
                                        'reading_at',
                                        F.when(
                                                 F.col('reading_at').isNotNull(),
                                                 F.date_format(
                                                               F.coalesce(
                                                                          F.try_to_date(F.col('reading_at'), 'dd/MM/yyyy'),
                                                                          F.try_to_date(F.col('reading_at'), 'yyyy-MM-dd')
                                                                         ),
                                                               'yyyy-MM-dd'
                                                              )
                                               )
                                             .otherwise(F.date_format(F.last_day(F.col('generated_at')), 'yyyy-MM-dd'))
                                      )
                           .withColumn(
                                        'consume_kwh',
                                        F.round(F.regexp_replace(F.col('consume_mwh'), ',', '.').cast('double') * F.lit(1000),0)
                                      )
                           .withColumn(
                                        'compensated_energy_kwh',
                                        F.round(F.regexp_replace(F.col('compensated_energy_mwh'), ',', '.').cast('double') * F.lit(1000),0)
                                      )
                           .withColumn(
                                        'compensated_energy_kwh',
                                        F.abs(F.col('compensated_energy_kwh'))
                                      )
                           .withColumn(
                                        'balance_kwh',
                                        F.round(F.regexp_replace(F.col('balance_mwh'), ',', '.').cast('double') * F.lit(1000),0)
                                      )
                           .withColumn('regional', F.lit(reg))
                           .select(
                                    'generated_at',
                                    'reading_at',
                                    'regional',
                                    'uc',
                                    'consume_kwh',
                                    'compensated_energy_kwh',
                                    'balance_kwh',
                                    '_source_file',
                                    '_source_path',
                                    '_ingestion_ts'
                                  )
                  )
        elif 'unidade_consumidora' in df_bronze.columns:
            # ---- CSV format (regional: sul) ----
            df = (
                    df_bronze.withColumnsRenamed(
                                                  {
                                                    'unidade_consumidora': 'uc',
                                                    'mes_referencia': 'generated_at',
                                                    'data_leitura': 'reading_at',
                                                    'consumo_kwh': 'consume_kwh',
                                                    'compensado_kwh': 'compensated_energy_kwh',
                                                    'saldo_kwh': 'balance_kwh',
                                                  }
                                                )
                             .withColumn(
                                          'generated_at',
                                          F.date_format(
                                                        F.to_date(
                                                                  F.concat(
                                                                            F.col('generated_at'),
                                                                            F.lit('-01')
                                                                          ),
                                                                  'yyyy-MM-dd'
                                                                 ),
                                                        'yyyy-MM-dd'
                                                       )
                                        )
                             .withColumn(
                                          'reading_at',
                                          F.when(
                                                   F.col('reading_at').isNotNull(),
                                                   F.date_format(
                                                                 F.coalesce(
                                                                            F.try_to_date(F.col('reading_at'), 'dd/MM/yyyy'),
                                                                            F.try_to_date(F.col('reading_at'), 'yyyy-MM-dd')
                                                                           ),
                                                                 'yyyy-MM-dd'
                                                                )
                                               )
                                             .otherwise(F.date_format(F.last_day(F.col('generated_at')), 'yyyy-MM-dd'))
                                        )
                             .withColumn('consume_kwh', F.col('consume_kwh').cast('double'))
                             .withColumn('compensated_energy_kwh', F.abs(F.col('compensated_energy_kwh')).cast('double'))
                             .withColumn('balance_kwh', F.col('balance_kwh').cast('double'))
                             .withColumn('regional', F.lit(reg))
                             .select(
                                      'generated_at',
                                      'reading_at',
                                      'regional',
                                      'uc',
                                      'consume_kwh',
                                      'compensated_energy_kwh',
                                      'balance_kwh',
                                      '_source_file',
                                      '_source_path',
                                      '_ingestion_ts'
                                    )
                 )
        elif 'UC' in df_bronze.columns:
            # ---- CSV format (regional: sudeste) ----
            df = (
                    df_bronze.withColumnsRenamed(
                                                  {
                                                    'UC': 'uc',
                                                    'MES_REF': 'generated_at',
                                                    'DT_LEITURA': 'reading_at',
                                                    'CONSUMO_KWH': 'consume_kwh',
                                                    'ENERGIA_COMPENSADA_KWH': 'compensated_energy_kwh',
                                                    'SALDO_CREDITOS_KWH': 'balance_kwh',
                                                  }
                                                )
                             .withColumn(
                                          'generated_at',
                                          F.date_format(
                                                         F.to_date(
                                                                    F.concat(
                                                                              F.element_at(F.split(F.col('generated_at'), '/'), 2),
                                                                              F.lit('-'),
                                                                              F.element_at(F.split(F.col('generated_at'), '/'), 1),
                                                                              F.lit('-01')
                                                                            ),
                                                                    'yyyy-MM-dd'
                                                                  ),
                                                         'yyyy-MM-dd'
                                                       )
                                        )
                             .withColumn(
                                          'reading_at',
                                          F.when(
                                                   F.col('reading_at').isNotNull(),
                                                   F.date_format(
                                                                 F.coalesce(
                                                                            F.try_to_date(F.col('reading_at'), 'dd/MM/yyyy'),
                                                                            F.try_to_date(F.col('reading_at'), 'yyyy-MM-dd')
                                                                           ),
                                                                 'yyyy-MM-dd'
                                                                )
                                               )
                                             .otherwise(F.date_format(F.last_day(F.col('generated_at')), 'yyyy-MM-dd'))
                                        )
                             .withColumn('consume_kwh', F.col('consume_kwh').cast('double'))
                             .withColumn('compensated_energy_kwh', F.abs(F.col('compensated_energy_kwh')).cast('double'))
                             .withColumn('balance_kwh', F.col('balance_kwh').cast('double'))
                             .withColumn('regional', F.lit(reg))
                             .select(
                                      'generated_at',
                                      'reading_at',
                                      'regional',
                                      'uc',
                                      'consume_kwh',
                                      'compensated_energy_kwh',
                                      'balance_kwh',
                                      '_source_file',
                                      '_source_path',
                                      '_ingestion_ts'
                                    )
                 )
        else:
            # ---- JSON format (regional: co) ----
            df = (
                    df_bronze.withColumnsRenamed({'referencia': 'generated_at'})
                             .withColumn(
                                          'generated_at', 
                                          F.date_format(F.to_date(F.concat(F.col('generated_at'), F.lit('01')), 'yyyyMMdd'), 'yyyy-MM-dd')
                                        )
                             .withColumn('m', F.explode('medicoes'))
                             .drop('medicoes', 'origem', 'regional', 'referencia')
                             .withColumn('regional', F.lit(reg))
                             .withColumn('regional', F.regexp_replace(F.col('regional'), 'co', 'centro-oeste'))
                             .select(
                                      'generated_at',
                                      'regional',
                                      F.coalesce(
                                                  F.to_date(F.col('m.leitura')), 
                                                  F.last_day(F.col('generated_at'))
                                                ).alias('reading_at'),
                                      F.col('m.uc').alias('uc'),
                                      F.col('m.valores.consumo').alias('consume_kwh'),
                                      F.abs(F.col('m.valores.energia_compensada')).alias('compensated_energy_kwh'),
                                      F.col('m.valores.saldo').alias('balance_kwh'),
                                      '_source_file',
                                      '_source_path',
                                      '_ingestion_ts'
                                    )
                 )

        if df_month is None:
            df_month = df
        else:
            df_month = df_month.unionByName(df)

        print(f"✓ {table_origin}")

    if df_month is not None:
        # uc must have 10 digits — pad with leading zeros
        df_month = df_month.withColumn('uc', F.lpad(F.trim(F.col('uc').cast('string')), 10, '0'))

        # primary key is (uc, generated_at) — remove duplicates
        df_month = check_and_remove_duplicates(df_month, ['uc', 'generated_at'])

        # unique sequential invoice_id per record (format INV1234567) — assigned before quarantine,
        # continuous across months (offset accumulated from previously processed months)
        df_month = df_month.withColumn(
                                        'invoice_id',
                                        F.concat(
                                                   F.lit('INV'),
                                                   F.lpad(
                                                          (F.lit(invoice_seq_offset)
                                                           + F.row_number().over(Window.orderBy('regional', 'uc'))).cast('string'),
                                                          7,
                                                          '0'
                                                         )
                                                 )
                                      )
        # advance the offset by this month's total assigned ids (pre-quarantine count)
        invoice_seq_offset += df_month.count()

        # cases to put in quarantine
        df_quarantine = (
                          df_month.filter(
                                             (F.col('compensated_energy_kwh').isNull())
                                           | (F.col('compensated_energy_kwh') > F.col('consume_kwh'))
                                           | (
                                              ~F.col('uc').isin(
                                                  [row.uc for row in spark.read.table('silver.contracts_uc').select('uc').distinct().collect()]
                                                               )
                                             )
                                         )
                                  .withColumn(
                                                'motive',
                                                F.when(
                                                        (F.col('compensated_energy_kwh').isNull()),
                                                        'Compensated energy is null'
                                                      )
                                                 .when(
                                                        (F.col('compensated_energy_kwh') > F.col('consume_kwh')),
                                                        'Energy compensated > consumed'
                                                      )
                                                 .when(
                                                        (~F.col('uc').isin(
                                                            [row.uc for row in spark.read.table('silver.contracts_uc').select('uc').distinct().collect()]
                                                                          )
                                                        ),
                                                        'UC without contract'
                                                      )
                                             )
                        )
        # writing quarantine
        quarantine_destiny = f"quarantine.warning_measurements_{m}"
        df_quarantine.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(quarantine_destiny)
        print(f"\n✓ {quarantine_destiny} written ({df_quarantine.count()} rows)\n")

        #writing measurements (all rows — quarantine exclusion happens in step03 via join with quarantine tables)
        table_destiny = f"silver.measurements_{m}"
        df_month.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(table_destiny)
        print(f"\n✓ {table_destiny} written ({df_month.count()} rows)\n")
